# 🔄 第7周-Day3：任务编排与工作流

> 从单工具到 DAG 工作流：拓扑排序 + 并行执行模拟
> 核心概念：任务节点、依赖关系、定时触发、消息路由

**实验目标：**
1. 实现 DAG 工作流 + 拓扑排序执行
2. 对比串行 vs 拓扑排序（允许并行）的执行效率
3. 可视化工作流 DAG 图

In [ ]:
# 配置 matplotlib 中文显示
from matplotlib import font_manager
import matplotlib.pyplot as plt
import numpy as np

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print(f"中文字体配置完成: {font_name}")

## 实验1：DAG 工作流 + 拓扑排序

In [ ]:
from collections import defaultdict, deque

class Task:
    def __init__(self, name, duration=1.0):
        self.name = name
        self.duration = duration
        self.done = False

class WorkflowDAG:
    def __init__(self):
        self.tasks: dict[str, Task] = {}
        self.edges: dict[str, list[str]] = defaultdict(list)  # A -> [B, C] means A must finish before B, C

    def add_task(self, name, duration=1.0):
        self.tasks[name] = Task(name, duration)

    def add_edge(self, before, after):
        self.edges[before].append(after)

    def topological_sort(self):
        in_degree = {n: 0 for n in self.tasks}
        for src, dsts in self.edges.items():
            for d in dsts:
                in_degree[d] += 1
        queue = deque([n for n, d in in_degree.items() if d == 0])
        order = []
        while queue:
            node = queue.popleft()
            order.append(node)
            for nb in self.edges[node]:
                in_degree[nb] -= 1
                if in_degree[nb] == 0:
                    queue.append(nb)
        return order

# 构建工作流：数据采集 → 清洗 → 分析 → 生成报告
dag = WorkflowDAG()
tasks = [("数据采集", 2), ("数据清洗", 3), ("质量检查", 1), ("数据分析", 4), ("生成报告", 2), ("发送通知", 0.5)]
for name, dur in tasks:
    dag.add_task(name, dur)
dag.add_edge("数据采集", "数据清洗")
dag.add_edge("数据清洗", "质量检查")
dag.add_edge("数据清洗", "数据分析")
dag.add_edge("质量检查", "数据分析")
dag.add_edge("数据分析", "生成报告")
dag.add_edge("生成报告", "发送通知")

print("拓扑排序:", dag.topological_sort())

## 实验2：串行 vs 并行执行时间对比

In [ ]:
def simulate_serial(dag):
    order = dag.topological_sort()
    total = sum(dag.tasks[n].duration for n in order)
    return order, total

def simulate_parallel(dag):
    order = dag.topological_sort()
    in_degree = {n: 0 for n in dag.tasks}
    for src, dsts in dag.edges.items():
        for d in dsts:
            in_degree[d] += 1
    remaining = dict(in_degree)
    finish_time = {}
    import heapq
    ready = [(0, n) for n, d in remaining.items() if d == 0]
    heapq.heapify(ready)
    while ready:
        t, node = heapq.heappop(ready)
        actual_start = t
        finish_time[node] = actual_start + dag.tasks[node].duration
        for nb_node in dag.edges[node]:
            remaining[nb_node] -= 1
            if remaining[nb_node] == 0:
                heapq.heappush(ready, (finish_time[node], nb_node))
    return max(finish_time.values())

serial_time = simulate_serial(dag)[1]
parallel_time = simulate_parallel(dag)
speedup = serial_time / parallel_time

print(f"串行执行: {serial_time}s")
print(f"并行执行: {parallel_time:.1f}s")
print(f"加速比: {speedup:.2f}x")

## 实验3：工作流 DAG 可视化

In [ ]:
import networkx as nx

G = nx.DiGraph()
for name, dur in tasks:
    G.add_node(name, duration=dur)
for src, dsts in dag.edges.items():
    for d in dsts:
        G.add_edge(src, d)

pos = nx.spring_layout(G, seed=42)
fig, ax = plt.subplots(figsize=(10, 6))
node_colors = ["#4CAF50", "#2196F3", "#FF9800", "#9C27B0", "#E91E63", "#00BCD4"]
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=2000, ax=ax)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=10)
nx.draw_networkx_edges(G, pos, ax=ax, arrows=True, arrowsize=20, edge_color="gray")
# 标注执行时间
for (x, y), (_, dur) in zip(pos.values(), tasks):
    ax.text(x, y - 0.15, f"{dur}s", ha='center', fontsize=9, color="red")
ax.set_title("数字员工工作流 DAG")
ax.axis('off')
plt.tight_layout()
plt.show()